<a href="https://colab.research.google.com/github/OfRoses/Generative-Pre-Trained-Transformer-Nano/blob/main/Generative_Pre_Trained_Transformer_Nano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Carregando Dataset

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#carregando um dataset que contém tudo o que shakespeare escreveu

--2023-10-30 13:15:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2023-10-30 13:15:27 (15.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



Abre o Dataset

In [ ]:
with open('input.txt', 'r', encoding='utf=8') as f:
  #funcao open que por parametro aceita ler um txt usando enconding
    text = f.read()
    #le o txt

Tamanho do dataset

In [ ]:
print("tamanho do dataset em letras: ", len(text)) #lentext visualiza quantas letras o dataset tem

print("\n")
print(text[:100]) #printa as primeiras 100 letras

tamanho do dataset em letras:  1115394


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


Letras que aparecem no texto

In [ ]:
#todas as letras unicas que aparecem no dataset
chars = sorted(list(set(text)))
tamanho_vocab = len(chars)

print(" ".join(chars))
print("quantos caracteres unicos tem = ",tamanho_vocab) #a variavel tamanho vocab guarda os 65 caracteres unicos do dataset


   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
quantos caracteres unicos tem =  65


Tokenizer letra a letra

In [ ]:
#mapeando as letras para integer(usaremos integer aqui para poupar processamento)
stoi = { ch:i for i,ch in enumerate(chars)}  #aqui estamos passando de caracter para inteiro
itos = { i:ch for i,ch in enumerate(chars) } #aqui estamos passando de inteiro para caracter

encode = lambda s: [stoi[c] for c in s] #input de uma string e output de inteiros
decode = lambda l: ''.join([itos[i] for i in l]) #input de inteiros e output de caracteres

print(encode("hello mister")) #testando o encode em duas palavras aleatorias
print(decode(encode("hello mister"))) #testando o decoder em 2 palavras aleatorias

[46, 43, 50, 50, 53, 1, 51, 47, 57, 58, 43, 56]
hello mister


Atribuindo tokens ao dataset inteiro

In [ ]:
import torch #usando pytorch
data = torch.tensor(encode(text), dtype=torch.long) #aplicando variavel long aos dados usando tensor
print(data.shape, data.dtype) #print dos dados da database agora em tensores
print(data[:100]) #as primeiras 100 palavras em inteiros vao ficar assim

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Separando o dataset em treino e validacao

In [ ]:
n = int (0.9*len(data)) #90% treino, 10% validacao para datasets grandes
treino_data = data[:n] #a parte que foi designada como treino é atribuida a variavel treino_data
val_data = data[n:] #a parte que foi atribuida como validacao é atribuida a variavel val_data

Print de um Tensor com Block size 8 para exemplificação

In [ ]:
block_size = 8 #block size é extremamente importante pois é o vetor que contém X letras, no caso aqui 8
treino_data[:block_size+1] #esse exemplo nao necessariamente remete a uma frase especifica
#mas contem varios samples do texto em que essa ordem eh exibida
#assim o GPT fara inferencias

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

Teste de inferencias baseado no Tensor anterior

In [ ]:
x = treino_data[:block_size] #usa o block size do tensor anterior para testar inferencias
y = treino_data[1:block_size+1] #com o +1 ele pode inferir o proximo encode
for t in range(block_size): #teste de inferencia
  context = x[:t+1]
  target = y[t]
  print(f"quando o input é {context} espera-se: {target}") #contexto de algum vetor dentro do dataset e sua letra inferida

quando o input é tensor([18]) espera-se: 47
quando o input é tensor([18, 47]) espera-se: 56
quando o input é tensor([18, 47, 56]) espera-se: 57
quando o input é tensor([18, 47, 56, 57]) espera-se: 58
quando o input é tensor([18, 47, 56, 57, 58]) espera-se: 1
quando o input é tensor([18, 47, 56, 57, 58,  1]) espera-se: 15
quando o input é tensor([18, 47, 56, 57, 58,  1, 15]) espera-se: 47
quando o input é tensor([18, 47, 56, 57, 58,  1, 15, 47]) espera-se: 58


separando os batch para treino e validacao com seed fixa

In [ ]:
torch.manual_seed(1337) #mesma seed
batch_size = 4 #quantas sequencias independentes serao processadas em paralelo
block_size = 8 #output maximo para inferencias de uma vez so

def get_batch(split):
    #gerando um pequeno batch de data com inputs x e targets y
    data = treino_data if split == 'train' else val_data #pega o vetor de treino
    ix = torch.randint(len(data) - block_size, (batch_size,)) #passa parametros de tamanho para o tensor e atribui para ix
    x = torch.stack([data[i:i+block_size]for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    return x, y

xb, yb = get_batch('train') #print de um batch de treinamento
print('inputs: ') #inputs do batch
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('----------')

for b in range(batch_size): #dimensao do batch
  for t in range(block_size): #dilacao do tempo, ou quantas vezes vai ser iterado cada batch
    context = xb[b, :t+1]
    target = yb[b,t]
  print(f'quando o input é {context.tolist()} espera-se: {target}') #input do contexto ou localizacao do vetor e sua proxima letra inferida, o target

inputs: 
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets: 
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----------
quando o input é [24, 43, 58, 5, 57, 1, 46, 43] espera-se: 39
quando o input é [44, 53, 56, 1, 58, 46, 39, 58] espera-se: 1
quando o input é [52, 58, 1, 58, 46, 39, 58, 1] espera-se: 46
quando o input é [25, 17, 27, 10, 0, 21, 1, 54] espera-se: 39


exemplo de input do transformer

In [ ]:
print(xb) #input do transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


Modelo do transformer

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): #similar a uma markov chain, porém ele aproxima a probabilidade usando todas as palavras lidas anteriormente

    def __init__(self, tamanho_vocab):
        super().__init__()
        #cada token analisado no tempo X vai ler as possibilidades do proximo
        self.token_embedding_table = nn.Embedding(tamanho_vocab, tamanho_vocab) #aqui o word embedding pega as 65 letras e usa a tecnica de word embedding
                                                                                #para atribuir valores numericos relacionais a cada um

    def forward(self, idx, targets=None):

        #idx e targets sao inteiros
        logits = self.token_embedding_table(idx) #Batch = 4 Time = 8 Channel = 65
        #logits referem-se a um tipo de tecnica de decoding não normalizado
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape #aqui é definido a forma do logits, ou qual informação o bloco principal de informação da llm vai ter
            logits = logits.view(B*T, C) #aqui pode-se ver que a variavel logits recebe multiplicações das matrizes referentes aos outros vetores
            targets = targets.view(B*T) #a variavel target se refere as inferencias ou as letras que vao aparecer depois
            loss = F.cross_entropy(logits, targets) #cross entropy é uma forma de descida do gradiente para calcular o erro

        return logits, loss


    #a parte que gera o modelo
    def generate(self, idx, max_new_tokens):
        # idx é um array dos batchs e block_size que é iterado 4 vezes
        for _ in range(max_new_tokens):
            # faz as inferencias
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # é transformado para caracter e contém a informações de (B, C)
            # Usa softmax para obter probabilidades
            probs = F.softmax(logits, dim=-1)#calcula a probabilidade da saida usando (B, C)
            # sample vindo da distribuicao de probabilidades calculada pela  usando (B, 1)
            idx_next = torch.multinomial(probs, num_samples=1)
            # inferencia a partir do batch analisado
            idx = torch.cat((idx, idx_next), dim=1) #aqui com o +1 podemos ter a inferencia obtida anteriormente (B, T+1)
        return idx

m = BigramLanguageModel(tamanho_vocab) #aqui o modelo recebe o tamanho da variavel tamanho_vocab
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


Otimizador

In [ ]:
# Criando um otimizador pelo Pytorch
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #AdamW regressor por descida do gradiente

Tamanho do batch, Exemplo de um batch e calculo da regressão

In [ ]:
batch_size = 32
for steps in range(100): # mais passos melhor resultado

    # exemplo de um batch de treino
    xb, yb = get_batch('train')

    # calculando a loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.587916374206543


Decoder dos tensor flow

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist())) #maximo de novos tokens gerados para print


xiKi-RJ:CgqVuUa!U?qMH.uk!sCuMXvv!CJFfx;LgRyJknOEti.?I&-gPlLyulId?XlaInQ'q,lT$
3Q&sGlvHQ?mqSq-eON
x?SP fUAfCAuCX:bOlgiRQWN:Mphaw
tRLKuYXEaAXxrcq-gCUzeh3w!AcyaylgYWjmJM?Uzw:inaY,:C&OECW:vmGGJAn3onAuMgia!ms$Vb q-gCOcPcUhOnxJGUGSPJWT:.?ujmJFoiNL&A'DxY,prZ?qdT;hoo'dHooXXlxf'WkHK&u3Q?rqUi.kz;?Yx?C&u3Qbfzxlyh'Vl:zyxjKXgC?
lv'QKFiBeviNxO'm!Upm$srm&TqViqiBD3HBP!juEOpmZJyF$Fwfy!PlvWPFC
&WDdP!Ko,px
x
tREOE;AJ.BeXkylOVD3KHp$e?nD,.SFbWWI'ubcL!q-tU;aXmJ&uGXHxJXI&Z!gHRpajj;l.
pTErIBjx;JKIgoCnLGXrJSP!AU-AcbczR?


Parte do processamento em Paralelo dos batchs

In [ ]:
class LayerNorm1d: #antes do 2.0 era BatchNorm1d

  def __init__(self, dim, eps=1e-5, momentum=0.1): #1e-5 é 1x10 elevado a menos 5
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # foward propagation
    xmean = x.mean(1, keepdim=True) # media do batch
    xvar = x.var(1, keepdim=True) # variancia do batch
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalizacao do batch
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self): #funcao de retorno necessaria
    return [self.gamma, self.beta]

torch.manual_seed(1337) ####### Mudança de Seed ######
module = LayerNorm1d(100)
x = torch.randn(32, 100) # tamanho do batch 32 de 100 vetores
x = module(x)
x.shape

torch.Size([32, 100])

Media dos batchs

In [ ]:
x[:,0].mean(), x[:,0].std() # media de todos os batchs

(tensor(0.1469), tensor(0.8803))

Caracteristicas de um token dentro do Batch

In [ ]:
x[0,:].mean(), x[0,:].std() # media das caracteristicas de um unico individuo dentro do batch

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
from google.colab import drive #acesso ao dataset no Google Drive
drive.mount('/content/drive/')

Mounted at /content/drive/


GPT -  VERSÃO 0.1

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparametros
batch_size = 16 # quantas sequencias serao processadas em paralelo
block_size = 32 # o maximo do tamanho do vetor que sera inferido
max_iters = 50000 #maximo de iterações do modelo
eval_interval = 100 #intervalo de avaliação
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu' #treino com GPU NVidia
eval_iters = 200
n_embd = 64 #quantos tokens serao usados
n_head = 4  #quantos tokens serão iniciados
n_layer = 4 #quantos layers temos no modelo
dropout = 0.0 #nao ignora nenhum calculo dos neuronios
# ------------

torch.manual_seed(1337)

!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# todos os caracteres unicos do .txt
chars = sorted(list(set(text)))
vocab_size = len(chars)
# criando um mapa de cahracter para inteiro e de inteiro para caracter
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # input de uma string e output de um vetor int
decode = lambda l: ''.join([itos[i] for i in l]) # input de vetor int output de caracter

# separacao de treino e teste
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # 90% treino 10%teste
train_data = data[:n]
val_data = data[n:]

# carregando os dados
def get_batch(split):
    # gerando um pequeno batch
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
#funcao de regressao que estima a loss
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):


    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        # B = Batch, T = Time C = Caracter, as formulas abaixo mostram quais variaveis as linhas de comando estao mexendo
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # computa os scores de acertos e os insere nos peseos
        #as formulas abaixo representam as multiplicações de matrizes que estão sendo feitas, com seu input e output
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # agrega os valores calculados para os pesos
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ Self Attention """

    def __init__(self, num_heads, head_size): #a função de self attention vai gerar através de 3 vetores as correlações entre as palavras
        super().__init__()                    #sendo similar a o que um word embedding faz
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ aqui temos um layer linear seguido de um não linear """

    def __init__(self, n_embd): #aqui temos os valores do word embedding sendo multiplicados pela rede neural e calculados pela relu
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), #aqui temos uma entrada lienar multiplicando por 4
            nn.ReLU(), #ativação relu
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    #atribui por word embedding as correlações da classe

    def __init__(self, n_embd, n_head):
        # Word Embedding
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd) #feed foward passa pelo vetor atribuindo cada letra seu valor numerico
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# Modelo de linguagem Bigram
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # cada token le o proximo que sera a inferencia da tabela atribuida pela markov chain
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #valor do token da letra
        self.position_embedding_table = nn.Embedding(block_size, n_embd) #posição dentro do vetor da letra
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # normalizacao do ultimo layer da camada
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        #idx e os alvos de inferencia sao os batchs e timers dos tensores inteiros do word embedding
        #B = Batch, T = Time, C = caracter, as formulas mostram quais informações estao inseridas nas matrizes atribuidas as variaveis
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    #esta é a parte que gera o texto de fato a partir do processamento dos inteiros do word embedding anterior
    def generate(self, idx, max_new_tokens):
        # a var idx é um array de indices do que está sendo processado
        for _ in range(max_new_tokens):
            # aqui o idx analisa o ultimo block do batch
            idx_cond = idx[:, -block_size:]
            # aqui o logits juntamente com o loss faz a predição de quais caracteres mostrar na tela
            logits, loss = self(idx_cond)
            # aqui ele faz o ultimo passo antes de converter de volta para caracter
            logits = logits[:, -1, :] # nesse momento ele volta a ser texto, com caracteres
            # aqui aplica-se softmax para obter as probabilidades
            probs = F.softmax(logits, dim=-1) # (B, C)
            # aqui se coleta um pedaço pequeno do que está sendo inferido para validar com a softmax
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # aqui é onde de fato o codigo irá fazer um loop para criar os textos
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# aqui temos um print de quantos parametros temos no modelo
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# aqui se cria o otimizador Adam pelo pytorch
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # dado intervalo X ele avalia o quanto está acertando, existe um delay para isso não prejudicar a performance
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #  aqui temos uma amostra dos dados de treino para inserir no algoritmo
    xb, yb = get_batch('train')

    # aqui se calcula quanto o modelo errou para usar na regressão
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# aqui se printa o que foi gerado pelo modelo depois de todo o treino e validação
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

--2023-10-18 11:01:52--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2023-10-18 11:01:52 (15.5 MB/s) - ‘input.txt’ saved [1115394/1115394]

0.209729 M parameters
step 0: train loss 4.4116, val loss 4.4022
step 100: train loss 2.6568, val loss 2.6670
step 200: train loss 2.5091, val loss 2.5059
step 300: train loss 2.4194, val loss 2.4335
step 400: train loss 2.3500, val loss 2.3564
step 500: train loss 2.2965, val loss 2.3127
step 600: train loss 2.2412, val loss 2.2502
step 700: train loss 2.2047, val loss 2.2183
step 800: train loss 

KeyboardInterrupt: ignored

GPT - VERSAO 0.2

In [ ]:
#importação inicial das biliotecas pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F

    ###########-Parametros e configurações ajustáveis do GPT-################

torch.manual_seed(6969) #Seed especifica, pode ser alterado
batch_size = 16 # quantas sequencias serao processadas em paralelo
block_size = 32 # o maximo do tamanho do vetor que sera inferido
max_iters = 5000 #maximo de iterações do modelo
eval_interval = 100 #intervalo de avaliação
learning_rate = 1e-3 #1 x 10 elevado a -3 - taxa de aprendizado por iteração
eval_iters = 200 #atualização dos intervalos do aprendizado
n_embd = 1000 #quantos tokens serao usados
n_head = 4  #quantas camadas de transformação serão iniciadas
n_layer = 4 #quantos layers temos no modelo
dropout = 0.0 #nao ignora nenhum calculo dos neuronios

    ###########----------------------------------------------################

     ###########- Inserção das databases para treino do GPT -################

     !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt #DATABASE shakespeare

     #!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz DATABASE imdb word-level
     #!tar -xf aclImdb_v1.tar.gz

     from google.colab import drive #acesso ao dataset no Google Drive
     drive.mount('/content/drive/')


     ###########----------------------------------------------###############

device = 'cuda' if torch.cuda.is_available() else 'cpu' #alocação de CUDA core para processamento paralelo em GPU NVidia

with open('input.txt', 'r', encoding='utf=8') as f:
  #funcao open que por parametro aceita ler um txt usando enconding
    text = f.read()
    #le o txt

    print("tamanho do dataset em letras: ", len(text)) #lentext visualiza quantas letras o dataset tem

print("\n\n\n")
print(text[:100]) #printa as primeiras 100 letras

#todas as letras unicas que aparecem no dataset
chars = sorted(list(set(text))) #sort dos caracteres
tamanho_vocab = len(chars) #variável tamanho_vocab recebe o vetor de quantos caracteres tem no dataset

print(" ".join(chars))
print("quantos caracteres unicos tem = \n\n",tamanho_vocab) #a variavel tamanho vocab guarda os 65 caracteres unicos do dataset

#mapeando as letras para integer(usaremos integer aqui para poupar processamento)
stoi = { ch:i for i,ch in enumerate(chars)}  #aqui estamos passando de caracter para inteiro
itos = { i:ch for i,ch in enumerate(chars) } #aqui estamos passando de inteiro para caracter

encode = lambda s: [stoi[c] for c in s] #input de uma string e output de inteiros
decode = lambda l: ''.join([itos[i] for i in l]) #input de inteiros e output de caracteres

print(encode("hello mister")) #testando o encode em duas palavras aleatorias
print(decode(encode("hello mister\n\n\n"))) #testando o decoder em 2 palavras aleatorias

data = torch.tensor(encode(text), dtype=torch.long) #aplicando variavel long aos dados usando tensor
print(data.shape, data.dtype) #print dos dados da database agora em tensores
print(data[:100]) #as primeiras 100 palavras em inteiros vao ficar assim

n = int (0.5*len(data)) #50% treino, 50% validacao para datasets grandes
treino_data = data[:n] #a parte que foi designada como treino é atribuida a variavel treino_data
val_data = data[n:] #a parte que foi atribuida como validacao é atribuida a variavel val_data

treino_data[:block_size+1] #esse exemplo nao necessariamente remete a uma frase especifica
#mas contem varios samples do texto em que essa ordem eh exibida
#assim o GPT fara inferencias

x = treino_data[:block_size] #usa o block size do tensor anterior para testar inferencias
y = treino_data[1:block_size+1] #com o +1 ele pode inferir o proximo encode
for t in range(block_size): #teste de inferencia
  context = x[:t+1] #printa a proxima letra
  target = y[t] #a letra que deve ser inferida é atribuida a target
  print(f"quando o input é {context} espera-se: {target}\n\n\n") #contexto de algum vetor dentro do dataset e sua letra inferida

def get_batch(split):
    #gerando um pequeno batch de data com inputs x e targets y
    data = treino_data if split == 'train' else val_data #pega o vetor de treino
    ix = torch.randint(len(data) - block_size, (batch_size,)) #passa parametros de tamanho para o tensor e atribui para ix
    x = torch.stack([data[i:i+block_size]for i in ix]) #atribuição dos parametros do torch para x e y
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    return x, y

@torch.no_grad()
def estimate_loss(): #função de perda pra calcular os erros do modelo
    out = {}
    model.eval()
    for split in ['train', 'val']: #divide a funcao de perda para treino e validacao
        losses = torch.zeros(eval_iters) #a perde é calculada a cada iteração
        for k in range(eval_iters):
            X, Y = get_batch(split) #calcula e perda a partir do batch
            logits, loss = model(X, Y) #e a partir dos logits
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
#Classe head que pega o numero do Word Embedding e aplica um ponteiro que inicia uma inferencia

 def __init__(self, head_size): #função head baseada no artigo ''attention is all you need''
        super().__init__()
        #3 parametros que atribuem valores aos tokens do word embedding, sendo a primeira etapa
        self.key = nn.Linear(n_embd, head_size, bias=False) #modelo linear de atribuição de valor numero a palavra
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

 def forward(self, x):
        B,T,C = x.shape
        # B = Batch, T = Time C = Caracter, as formulas abaixo mostram quais variaveis as linhas de comando estao mexendo
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # computa os scores de acertos e os insere nos peseos
        #as formulas abaixo representam as multiplicações de matrizes que estão sendo feitas, com seu input e output
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T) #transformação da matriz em transposta
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T) #se a matriz ficar incompleta preenche com um 0
        wei = F.softmax(wei, dim=-1) # (B, T, T) #softmax em cada block_size pra calcular o peso
        wei = self.dropout(wei) #dropout de pesos aleatorios
        # agrega os valores calculados para os pesos
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C) #transformação da matriz em caracter
        return out

class MultiHeadAttention(nn.Module):
    #self Attention

    def __init__(self, num_heads, head_size): #a função de self attention vai gerar através de 3 vetores as correlações entre as palavras
        super().__init__()                    #sendo similar a o que um word embedding faz
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    #aqui temos um layer linear seguido de um não linear

    def __init__(self, n_embd): #aqui temos os valores do word embedding sendo multiplicados pela rede neural e calculados pela relu
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), #aqui temos uma entrada lienar multiplicando por 4
            nn.ReLU(), #Função de ativação ReLU que zera valores negativos
            nn.Linear(4 * n_embd, n_embd), #multiplicação linear
            nn.Dropout(dropout), #parametro dropout zero setada por hyperparametro
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    #atribui por word embedding as correlações da classe head

    def __init__(self, n_embd, n_head):
        # Word Embedding
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd) #feed foward passa pelo vetor atribuindo cada letra seu valor numerico
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

xb, yb = get_batch('train') #print de um batch de treinamento
print('inputs: ') #inputs do batch
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('----------\n\n\n')

for b in range(batch_size): #dimensao do batch
  for t in range(block_size): #dilacao do tempo, ou quantas vezes vai ser iterado cada batch
    context = xb[b, :t+1]
    target = yb[b,t]
  print(f'quando o input é {context.tolist()} espera-se: {target}\n\n\n') #input do contexto ou localizacao do vetor e sua proxima letra inferida, o target

print(xb) #input do transformer

# Modelo de linguagem Bigram
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # cada token le o proximo que sera a inferencia da tabela atribuida pela markov chain
        self.token_embedding_table = nn.Embedding(tamanho_vocab, n_embd) #valor do token da letra
        self.position_embedding_table = nn.Embedding(block_size, n_embd) #posição dentro do vetor da letra
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # normalizacao do ultimo layer da camada
        self.lm_head = nn.Linear(n_embd, tamanho_vocab)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        #idx e os alvos de inferencia sao os batchs e timers dos tensores inteiros do word embedding
        #B = Batch, T = Time(), C = caracter, as formulas mostram quais informações estao inseridas nas matrizes atribuidas as variaveis
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) posição da palavra dentro do contexto
        x = tok_emb + pos_emb # (B,T,C) usase 3 parametros da função head
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size) os logits recebem a atribuição do vetor vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape #logits recebe os 3 vetores
            logits = logits.view(B*T, C) #multiplicação de (B T)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    #esta é a parte que gera o texto de fato a partir do processamento dos inteiros do word embedding anterior
    def generate(self, idx, max_new_tokens):
        # a var idx é um array de indices do que está sendo processado
        for _ in range(max_new_tokens):
            # aqui o idx analisa o ultimo block do batch
            idx_cond = idx[:, -block_size:]
            # aqui o logits juntamente com o loss faz a predição de quais caracteres mostrar na tela
            logits, loss = self(idx_cond)
            # aqui ele faz o ultimo passo antes de converter de volta para caracter
            logits = logits[:, -1, :] # nesse momento ele volta a ser texto, com caracteres
            # aqui aplica-se softmax para obter as probabilidades
            probs = F.softmax(logits, dim=-1) # (B, C)
            # aqui se coleta um pedaço pequeno do que está sendo inferido para validar com a softmax
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # aqui é onde de fato o codigo irá fazer um loop para criar os textos
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel() #aqui o modelo recebe o tamanho da variavel tamanho_vocab
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

# Criando um otimizador pelo Pytorch
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #AdamW regressor por descida do gradiente

for steps in range(100): # mais passos melhor resultado

    # exemplo de um batch de treino
    xb, yb = get_batch('train')

    # calculando a perda
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward() #perda similar a retropropagação
    optimizer.step()

print(loss.item())

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist())) #maximo de novos tokens gerados para print

class LayerNorm1d: #antes do 2.0 era BatchNorm1d

  def __init__(self, dim, eps=1e-5, momentum=0.1): #1e-5 é 1x10 elevado a menos 5
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # propagação para frente do modelo (foward propagation)
    xmean = x.mean(1, keepdim=True) # media do batch
    xvar = x.var(1, keepdim=True) # variancia do batch
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalizacao do batch
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self): #funcao de retorno necessaria
    return [self.gamma, self.beta]

module = LayerNorm1d(100)
x = torch.randn(32, 100) # tamanho do batch 32 de 100 vetores
x = module(x)
x.shape

x[:,0].mean(), x[:,0].std() # media de todos os batchs, normalização

x[0,:].mean(), x[0,:].std() # media das caracteristicas de um unico individuo dentro do batch

model = BigramLanguageModel() #chamada da função do modelo Bigram
m = model.to(device)
# aqui temos um print de quantos parametros temos no modelo
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# aqui se cria o otimizador Adam pelo pytorch
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # dado intervalo X ele avalia o quanto está acertando, existe um delay para isso não prejudicar a performance
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss() #estimação da perda do GPT
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #  aqui temos uma amostra dos dados de treino para inserir no algoritmo
    xb, yb = get_batch('train')

    # aqui se calcula quanto o modelo errou para usar na regressão
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# aqui se printa o que foi gerado pelo modelo depois de todo o treino e validação
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
#max new tokens é o quão grande o testo gerado vai ser

--2023-10-18 11:15:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.07s   

2023-10-18 11:15:01 (15.7 MB/s) - ‘input.txt.2’ saved [1115394/1115394]

tamanho do dataset em letras:  1115394




First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

   ! $ & ' , - . 3 : ; ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z a b c d e f g h i j k l m n o p q r s t u v w x y z
quantos caracteres unicos tem = 

 65
[46, 43, 50, 50, 53, 1, 51, 47, 57, 58, 43, 56]
hello mister



torch.Size([1115394])